# Hopfield Network

A Hopfield network has one layer of $N$ neurons and is a fully interconnected recurrent neural network: each neuron is connected to every other neuron. After initialization of all neurons (the initial input), the network is let to evolve: an output at time $t$ becomes an input at time $t+1$. Thus to generate a series of outputs, we have to provide only one initial input. In the course of this dynamical evolution the network should reach a stable state (an attractor), which is a configuration of neuron values that is not changed by subsequent updates of the network. Networks of this kind are used as models of associative memory. After initialization the network should evolve to the closest attractor.

---

**This notebook provides a comprehensive exploration of Hopfield networks** using an enhanced `HopfieldNetwork` class with built-in visualization and analysis tools.

- **Exercise 1** serves as a **complete tutorial**: it walks through every available analysis tool on a simple 2D network so you learn the full toolkit.
- **Exercises 2–5** provide skeleton code for the minimum analysis. Apply the tools you learned in Exercise 1 to go deeper.

## Colab Setup
This part is only required when running this notebook "in the cloud" on [Google Colab](https://colab.research.google.com). When running it locally, skip this part and go to the next section.

### [OPTIONAL] Interactive Plots
Run the below cell if you want to enable interactive plots in Google Colab. For most exercises this is not required and it can be quite slow.

In [ ]:
!pip install ipympl
%matplotlib widget
from google.colab import output
output.enable_custom_widget_manager()

### [REQUIRED] Auxiliary Files

In [ ]:
# Retrieve the enhanced HopfieldNetwork class
!wget -q https://raw.githubusercontent.com/psanchezML/KUL_ANNDL_ExerciseSessions/main/session2/hopfield.py
print("hopfield.py downloaded successfully.")

## Setup
Import all the necessary modules used throughout this notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from hopfield import HopfieldNetwork

rng = np.random.default_rng(42)
%matplotlib inline

---
## Exercise 1: 2D Hopfield Network — Complete Toolkit Tutorial

We create the simplest non-trivial Hopfield network — **2 neurons, 3 target patterns** — and use it as a sandbox to demonstrate **every analysis and visualization tool** available in the `HopfieldNetwork` class.

> **Why this matters:** Once you master these tools on a 2D network where everything is easy to visualize, you can apply the same functions to higher-dimensional networks (3D, digits, etc.) in the following exercises.

**Target patterns:** $[1, 1]$, $[-1, -1]$, $[1, -1]$

### 1.1 Creating the Network and Inspecting its Configuration

The `summary()` method gives you a quick overview: dimension, number of stored patterns, algorithm used, theoretical capacity, and how many patterns are stable fixed points.

In [ ]:
targets_2d = np.array([[1, 1], [-1, -1], [1, -1]])

# Create the network (default algorithm: LSSM)
net_2d = HopfieldNetwork(targets_2d)
net_2d.summary()

In [ ]:
# You can also create a Hebb-rule network for comparison
net_2d_hebb = HopfieldNetwork(targets_2d, alg='Hebb')
net_2d_hebb.summary()

**Key observations:**
- The network has dimension $D=2$ and stores $T=3$ patterns.
- Theoretical Hebb capacity is $\sim 0.14 \times 2 = 0$ — we are **above capacity** even for this tiny network! (But LSSM can still handle it.)
- `summary()` tells you how many stored patterns are actually stable fixed points.

**Useful properties:**

In [ ]:
print(f"Targets shape:         {net_2d.targets.shape}")
print(f"Number of patterns:    {net_2d.num_patterns}")
print(f"Dimension:             {net_2d.D}")
print(f"Theoretical capacity:  {net_2d.theoretical_capacity}")
print(f"Weight matrix shape:   {net_2d.W.shape}")
print(f"Bias shape:            {net_2d.b.shape}")

### 1.2 Simulating State Evolution

`simulate(data, num_iter, sync)` runs the network dynamics and returns the **full trajectory** (all states) and **energies** at each step.

Let's simulate from several initial states:

In [ ]:
initial_states = np.array([
    [0.5, 0.3],     # near target [1,1]
    [-0.8, 0.7],    # ambiguous region
    [0.1, -0.9],    # near target [1,-1]
    [-0.3, -0.5],   # near target [-1,-1]
    [0.0, 0.0],     # origin — equidistant from all
    [-1.0, 1.0],    # corner not matching any target
])

for i, init in enumerate(initial_states):
    states, energies = net_2d.simulate(init, num_iter=15)
    final = states[:, -1]
    idx, dist = net_2d.nearest_target(final)
    print(f"  Init {init} → Final [{final[0]:+.2f}, {final[1]:+.2f}]"
          f"  |  Nearest target: T{idx} = {net_2d.targets[idx]}"
          f"  |  Hamming dist: {dist}"
          f"  |  Converged in ~{np.argmax(np.all(np.diff(states, axis=1) == 0, axis=0)) or 15} steps")

**Questions to reflect on:**
- Do all initial states converge to one of the 3 target patterns?
- Are there initial states that converge to a **non-target** attractor (a *spurious* attractor)?
- How many iterations does it typically take to converge?

### 1.3 Visualizing State Trajectories

`plot_state_evolution(states, targets)` draws the trajectory in state space. For 2D networks this is a scatter plot with arrows showing the path from initial to final state.

In [ ]:
# Pick two interesting trajectories
states_a, _ = net_2d.simulate([0.5, 0.3], num_iter=15)
states_b, _ = net_2d.simulate([-0.8, 0.7], num_iter=15)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
net_2d.plot_state_evolution(states_a, targets=targets_2d, ax=ax1)
ax1.set_title('Trajectory from (0.5, 0.3)')
net_2d.plot_state_evolution(states_b, targets=targets_2d, ax=ax2)
ax2.set_title('Trajectory from (-0.8, 0.7)')
plt.tight_layout()
plt.show()

### 1.4 Energy Landscape

`plot_energy_landscape(trajectories, resolution)` draws a **contour plot** of the energy function $E(s_1, s_2)$ over the entire state space $[-1,1]^2$.

Trajectories are overlaid to show how each initial state rolls downhill toward a minimum.

> This is the single most important visualization: it shows you **where the basins are**, **how deep** each energy well is, and **where spurious attractors** might hide.

In [ ]:
# Collect trajectories from all initial states
trajectories = []
for init in initial_states:
    st, _ = net_2d.simulate(init, num_iter=15)
    trajectories.append(st)

fig, ax = net_2d.plot_energy_landscape(trajectories=trajectories, resolution=100)
plt.show()

**What to look for:**
- The stored patterns (gold diamonds) should sit at **local minima** (dark regions).
- Trajectories should flow from light (high energy) to dark (low energy).
- Any local minimum that is NOT a gold diamond is a **spurious attractor**.

### 1.5 Energy Decay Over Time

`plot_energy_over_time(energies, labels)` shows how the energy decreases at each iteration for one or more trajectories.

> **Key insight:** The energy function is a Lyapunov function — it **never increases** during asynchronous updates (and typically doesn't increase during synchronous updates for symmetric weights).

In [ ]:
all_energies = []
labels = []
for i, init in enumerate(initial_states):
    _, en = net_2d.simulate(init, num_iter=15)
    all_energies.append(en)
    labels.append(f'({init[0]:+.1f}, {init[1]:+.1f})')

fig, ax = net_2d.plot_energy_over_time(np.array(all_energies), labels=labels)
plt.show()

**Observe:** All trajectories decrease monotonically. The final energy level tells you *which* basin the trajectory settled into — trajectories converging to the same attractor end at the same energy.

### 1.6 Weight Matrix Visualization

`plot_weight_matrix()` displays the learned weights as a heatmap. For small networks you can read off the individual connection strengths.

In [ ]:
fig, ax = net_2d.plot_weight_matrix()
plt.show()

print("Weight matrix W:")
print(net_2d.W)
print(f"\nBias b: {net_2d.b}")

**Interpretation:** Positive $w_{ij}$ means neurons $i$ and $j$ tend to have the same sign in the stored patterns. Negative $w_{ij}$ means they tend to have opposite signs. The diagonal is always zero (no self-connections).

### 1.7 Basins of Attraction

`plot_basins(resolution, num_iter, sync)` samples the entire state space, runs each point to convergence, and **color-codes** each region by which attractor it converges to.

Stars ($\star$) mark attractors. Percentages show the fraction of state space in each basin.

In [ ]:
fig, ax = net_2d.plot_basins(resolution=100, num_iter=30, sync=True)
plt.show()

**Questions:**
- How many distinct basins are there? Are there more than 3 (the number of stored patterns)?
- Which basin is the largest? Does it correspond to the "deepest" energy minimum?
- Do any regions converge to a spurious attractor that isn't one of the 3 target patterns?

### 1.8 Fixed Points and Spurious Attractors

Three key analysis functions:
- `is_stable(pattern)`: checks if a specific pattern is a fixed point
- `classify_fixed_points()`: discovers **all** fixed points by enumeration (feasible for $D \leq 20$)
- `find_spurious_attractors()`: filters fixed points to find those that are **not** stored patterns or their negations

In [ ]:
# Check stability of each target
print("Target stability check:")
print("-" * 50)
for i, t in enumerate(targets_2d):
    stable = net_2d.is_stable(t)
    e = net_2d.energy(t)
    print(f"  Target {i} = {t}:  stable = {stable},  energy = {e:.4f}")

# Discover ALL fixed points
print(f"\nAll fixed points of the network:")
print("-" * 50)
fps = net_2d.classify_fixed_points()
for i, fp in enumerate(fps):
    is_stored = any(np.allclose(fp, t, atol=0.5) for t in targets_2d)
    is_neg = any(np.allclose(fp, -t, atol=0.5) for t in targets_2d)
    stable = net_2d.is_stable(fp)
    label = "STORED" if is_stored else ("NEGATION" if is_neg else "SPURIOUS")
    print(f"  FP {i}: {fp}  [{label}]  stable={stable}  energy={net_2d.energy(fp):.4f}")

# Spurious attractors specifically
spurious = net_2d.find_spurious_attractors()
print(f"\nSpurious attractors found: {len(spurious)}")
for s in spurious:
    print(f"  {s}  energy={net_2d.energy(s):.4f}")

**Key concept:** Spurious attractors are a fundamental limitation of Hopfield networks. They arise because the energy landscape has local minima that don't correspond to any stored pattern. Common types include:
1. **Negation states**: $-\xi^\mu$ (the negative of a stored pattern)
2. **Mixture states**: linear combinations of stored patterns
3. **Spin-glass states**: unrelated to any stored pattern

### 1.9 Comprehensive Dashboard

`plot_dashboard(data, num_iter, sync, shape)` combines energy decay, weight matrix, state heatmap, and (for image data) reconstruction snapshots into a single figure.

For low-dimensional networks it shows initial vs. final state as a bar chart.

In [ ]:
fig = net_2d.plot_dashboard([0.5, 0.3], num_iter=15)
plt.show()

### 1.10 Multi-Trajectory Comparison

`plot_multi_trajectory(data_list, num_iter, sync, labels)` runs multiple initial states and compares:
- **Left panel**: energy curves overlaid
- **Right panel**: final neuron activations as grouped bars

In [ ]:
starts = [
    [0.5, 0.3],
    [-0.8, 0.7],
    [0.1, -0.9],
    [-0.3, -0.5],
    [0.0, 0.0],
]
fig, axes = net_2d.plot_multi_trajectory(
    starts, num_iter=20,
    labels=[f'({s[0]:+.1f}, {s[1]:+.1f})' for s in starts]
)
plt.show()

### 1.11 Synchronous vs Asynchronous Update

Compare how the update mode affects convergence. Asynchronous updates guarantee energy decrease at **every** neuron update (for symmetric weights), while synchronous updates may cause oscillations.

In [ ]:
init_test = np.array([0.6, -0.4])

# Synchronous
st_sync, en_sync = net_2d.simulate(init_test, num_iter=20, sync=True)
# Asynchronous
st_async, en_async = net_2d.simulate(init_test, num_iter=20, sync=False)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Energy comparison
axes[0].plot(en_sync, 'o-', ms=3, label='Synchronous', color='steelblue')
axes[0].plot(en_async, 's-', ms=3, label='Asynchronous', color='coral')
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Energy')
axes[0].set_title('Energy: Sync vs Async')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Trajectory comparison (state space)
axes[1].plot(st_sync[0], st_sync[1], 'o-', ms=4, label='Sync', color='steelblue')
axes[1].plot(st_async[0], st_async[1], 's-', ms=4, label='Async', color='coral')
for t in targets_2d:
    axes[1].plot(t[0], t[1], 'D', color='gold', ms=10, mec='black')
axes[1].set_xlim(-1.3, 1.3)
axes[1].set_ylim(-1.3, 1.3)
axes[1].set_title('Trajectories in State Space')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_aspect('equal')

# Final states
axes[2].bar([0, 1], st_sync[:, -1], 0.35, label='Sync final', color='steelblue', alpha=0.8)
axes[2].bar([0.35, 1.35], st_async[:, -1], 0.35, label='Async final', color='coral', alpha=0.8)
axes[2].set_xticks([0.175, 1.175])
axes[2].set_xticklabels(['Neuron 1', 'Neuron 2'])
axes[2].set_title('Final States')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Sync final:  [{st_sync[0,-1]:+.4f}, {st_sync[1,-1]:+.4f}]  energy={en_sync[-1]:.4f}")
print(f"Async final: [{st_async[0,-1]:+.4f}, {st_async[1,-1]:+.4f}]  energy={en_async[-1]:.4f}")

### 1.12 Hamming Distance and Nearest Target

- `hamming_distance(a, b)`: counts how many components differ in sign
- `nearest_target(state)`: returns the index and Hamming distance of the closest stored pattern

In [ ]:
test_state = np.array([0.8, -0.6])
final_state = net_2d._evolve(test_state.reshape(1, -1), num_iter=20)[0]

print(f"Test state: {test_state}")
print(f"Final state: {final_state}")
print()
for i, t in enumerate(targets_2d):
    d = HopfieldNetwork.hamming_distance(final_state, t)
    print(f"  Hamming distance to target {i} ({t}): {d}")

idx, dist = net_2d.nearest_target(final_state)
print(f"\nNearest target: T{idx} = {net_2d.targets[idx]}, distance = {dist}")

### 1.13 Algorithm Comparison: Hebb vs LSSM

Let's compare the basins for the same targets under both learning rules.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

net_hebb = HopfieldNetwork(targets_2d, alg='Hebb')
net_lssm = HopfieldNetwork(targets_2d, alg='LSSM')

net_hebb.plot_basins(resolution=80, num_iter=30, ax=ax1)
ax1.set_title('Basins of Attraction (Hebb)')

net_lssm.plot_basins(resolution=80, num_iter=30, ax=ax2)
ax2.set_title('Basins of Attraction (LSSM)')

plt.tight_layout()
plt.show()

print("\nHebb network:"); net_hebb.summary()
print("\nLSSM network:"); net_lssm.summary()

### Summary of Available Tools

You now know how to use every analysis and visualization function:

| Method | Purpose |
|:-------|:--------|
| `summary()` | Network configuration overview |
| `simulate(data, num_iter, sync)` | Run dynamics, get full trajectory + energies |
| `is_stable(pattern)` | Check if a pattern is a fixed point |
| `classify_fixed_points()` | Find all fixed points by enumeration |
| `find_spurious_attractors()` | Find attractors that aren't stored patterns |
| `compute_basins(...)` | Map initial states → attractors programmatically |
| `capacity_test(dim, max, trials)` | Automated capacity experiment |
| `hamming_distance(a, b)` | Bit-difference between two states |
| `nearest_target(state)` | Closest stored pattern |
| `plot_energy_landscape(...)` | Contour/surface plot of energy (D ≤ 3) |
| `plot_basins(...)` | Color-coded basin map |
| `plot_energy_over_time(...)` | Energy decay curves |
| `plot_state_evolution(...)` | Trajectory visualization |
| `plot_weight_matrix()` | Weight heatmap |
| `plot_dashboard(...)` | All-in-one view |
| `plot_multi_trajectory(...)` | Compare multiple initial states |
| `plot_pattern_reconstruction(...)` | Original / Noisy / Reconstructed grid |
| `plot_capacity_analysis(...)` | Retrieval rate vs. pattern count |

**Apply these freely in the following exercises!**

---
## Exercise 2: 3D Hopfield Network

Create a three-neuron Hopfield network with target patterns $[1, 1, 1]$, $[-1, -1, 1]$ and $[1, -1, -1]$.

**Minimum required analysis** (skeleton code provided below):
1. Create the network and check stability of all targets
2. Simulate from a few initial states and note which attractors they converge to
3. Visualize energy landscape slices and basins of attraction

**For deeper analysis**, apply the tools you learned in Exercise 1: multi-trajectory comparison, sync vs async, spurious attractor search, etc.

> **Reflective questions** (answer in your report):
> - Are all three target patterns stable fixed points? If not, which ones are unstable and why?
> - How does the 3D energy landscape differ qualitatively from the 2D case?
> - Compare basins under synchronous vs. asynchronous update — do they differ?
> - Can you find any spurious attractors? Where do they sit in state space relative to the stored patterns?
> - What happens when you start from the origin $[0, 0, 0]$?

In [ ]:
# --- 2.1 Create the network ---
targets_3d = np.array([[1, 1, 1], [-1, -1, 1], [1, -1, -1]])
net_3d = HopfieldNetwork(targets_3d)
net_3d.summary()

# Check stability of each target
for i, t in enumerate(targets_3d):
    print(f"  Target {i} = {t}:  stable = {net_3d.is_stable(t)},  energy = {net_3d.energy(t):.4f}")

In [ ]:
# --- 2.2 Simulate from selected initial states ---
test_inputs_3d = [
    [0.5, 0.5, 0.5],
    [-0.5, -0.5, 0.5],
    [0.5, -0.5, -0.5],
    [0.0, 0.0, 0.0],
    [-1.0, 1.0, 1.0],
]

for init in test_inputs_3d:
    states, energies = net_3d.simulate(init, num_iter=20)
    final = states[:, -1]
    idx, dist = net_3d.nearest_target(final)
    print(f"  {init} → [{final[0]:+.2f}, {final[1]:+.2f}, {final[2]:+.2f}]"
          f"  nearest=T{idx}  hamming={dist}")

In [ ]:
# --- 2.3 Energy landscape slices ---
fig, _ = net_3d.plot_energy_landscape(resolution=60)
plt.show()

In [ ]:
# --- 2.4 Basins of attraction ---
fig, ax = net_3d.plot_basins(resolution=15, num_iter=30, sync=True)
plt.show()

In [ ]:
# --- 2.5 YOUR ANALYSIS ---
# Apply tools from Exercise 1 for deeper investigation.
# Suggestions:
#   - net_3d.classify_fixed_points()
#   - net_3d.find_spurious_attractors()
#   - net_3d.plot_multi_trajectory(test_inputs_3d, num_iter=20)
#   - net_3d.plot_dashboard([0.0, 0.0, 0.0], num_iter=20)
#   - Compare sync vs async basins


---
## Exercise 3: Handwritten Digit Reconstruction

We train a 784-dimensional Hopfield network to store MNIST digits and test its ability to reconstruct noisy inputs.

**This section provides comprehensive ablation tools** — not just a slider, but systematic experiments that measure reconstruction quality across noise levels, digit subsets, and algorithms.

### 3.1 Load and Prepare Data

In [ ]:
from keras.datasets import mnist as mnist_data
(digits_raw, labels_raw), _ = mnist_data.load_data()

# Select one representative per digit class
digit_images = {}
for d in range(10):
    idx = np.where(labels_raw == d)[0][0]
    digit_images[d] = digits_raw[idx]

# Normalize to bipolar {-1, +1}
all_targets = np.array([digit_images[d] for d in range(10)])
all_targets_norm, norm_params = HopfieldNetwork.normalize(all_targets)

print(f"Digit shape: {digit_images[0].shape}")
print(f"Normalized shape: {all_targets_norm.shape}")
print(f"Value range: [{all_targets_norm.min()}, {all_targets_norm.max()}]")

### 3.2 Store All 10 Digits and Reconstruct

In [ ]:
net_digits = HopfieldNetwork(all_targets_norm)
net_digits.summary()

In [ ]:
# Reconstruct all 10 digits from noisy inputs
noise_level = 1.0
noisy = np.clip(
    all_targets_norm + rng.normal(scale=noise_level, size=all_targets_norm.shape),
    -1, 1
)
states_all, energies_all = net_digits.simulate(noisy, num_iter=30)

# Reconstruction comparison grid
reconstructed = states_all[:, :, -1]
fig, axes = HopfieldNetwork.plot_pattern_reconstruction(
    all_targets_norm, noisy, reconstructed, shape=(28, 28)
)
plt.show()

### 3.3 Noise Sensitivity Analysis

Systematically sweep noise levels and measure reconstruction quality (Hamming distance to nearest target) for each digit.

In [ ]:
noise_levels = np.arange(0.0, 5.1, 0.5)
num_iter_test = 30
n_repeats = 5

# Results: avg Hamming distance per noise level
avg_hamming = np.zeros((len(noise_levels), 10))
retrieval_accuracy = np.zeros(len(noise_levels))

for ni, noise in enumerate(noise_levels):
    total_correct = 0
    for rep in range(n_repeats):
        noisy_rep = np.clip(
            all_targets_norm + rng.normal(scale=noise, size=all_targets_norm.shape),
            -1, 1
        )
        states_rep, _ = net_digits.simulate(noisy_rep, num_iter=num_iter_test)
        final_rep = states_rep[:, :, -1]
        for d in range(10):
            idx, dist = net_digits.nearest_target(final_rep[d])
            avg_hamming[ni, d] += dist / n_repeats
            if idx == d:
                total_correct += 1
    retrieval_accuracy[ni] = total_correct / (10 * n_repeats)

# Plot 1: Average Hamming distance per digit vs noise
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for d in range(10):
    ax1.plot(noise_levels, avg_hamming[:, d], 'o-', ms=3, label=f'Digit {d}')
ax1.set_xlabel('Noise Level (σ)')
ax1.set_ylabel('Avg Hamming Distance to Nearest Target')
ax1.set_title('Reconstruction Quality vs Noise')
ax1.legend(fontsize=7, ncol=2)
ax1.grid(True, alpha=0.3)

# Plot 2: Overall retrieval accuracy vs noise
ax2.plot(noise_levels, retrieval_accuracy * 100, 'o-', lw=2, ms=5, color='steelblue')
ax2.axhline(y=100, color='green', ls=':', alpha=0.5, label='Perfect')
ax2.axhline(y=10, color='red', ls=':', alpha=0.5, label='Random chance')
ax2.set_xlabel('Noise Level (σ)')
ax2.set_ylabel('Correct Retrieval (%)')
ax2.set_title('Retrieval Accuracy vs Noise')
ax2.set_ylim(-5, 110)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nRetrieval accuracy at σ=0.0: {retrieval_accuracy[0]*100:.0f}%")
print(f"Retrieval accuracy at σ=1.0: {retrieval_accuracy[2]*100:.0f}%")
print(f"Retrieval accuracy at σ=3.0: {retrieval_accuracy[6]*100:.0f}%")

### 3.4 Per-Digit Dashboard

Inspect the full reconstruction process for a specific digit using the dashboard view.

In [ ]:
digit_to_inspect = 3
noise_for_dashboard = 1.5

noisy_single = np.clip(
    all_targets_norm[digit_to_inspect] + rng.normal(scale=noise_for_dashboard, size=784),
    -1, 1
)
fig = net_digits.plot_dashboard(noisy_single, num_iter=30, shape=(28, 28))
fig.suptitle(f'Dashboard: Digit {digit_to_inspect} (noise σ={noise_for_dashboard})', fontsize=14)
plt.show()

In [ ]:
# Energy evolution for all 10 digits at once
noise_energy = 1.0
noisy_en = np.clip(
    all_targets_norm + rng.normal(scale=noise_energy, size=all_targets_norm.shape), -1, 1
)
_, energies_en = net_digits.simulate(noisy_en, num_iter=30)

fig, ax = net_digits.plot_energy_over_time(
    energies_en, labels=[f'Digit {i}' for i in range(10)]
)
ax.set_title(f'Energy Decay for All 10 Digits (noise σ={noise_energy})')
plt.show()

### 3.5 Storing Subsets of Digits

The theoretical capacity for 784 neurons is $\sim 0.14 \times 784 \approx 110$ patterns, so storing 10 digits is well within capacity. But what if we store fewer? Does reconstruction improve?

In [ ]:
subsets = [
    [0, 1],
    [0, 1, 2, 3, 4],
    list(range(10)),
]

fig, axes = plt.subplots(1, len(subsets), figsize=(6 * len(subsets), 5))

noise_sub = 1.5
for si, subset in enumerate(subsets):
    targets_sub = all_targets_norm[subset]
    net_sub = HopfieldNetwork(targets_sub)
    noisy_sub = np.clip(
        targets_sub + rng.normal(scale=noise_sub, size=targets_sub.shape), -1, 1
    )
    states_sub, _ = net_sub.simulate(noisy_sub, num_iter=30)
    recon_sub = states_sub[:, :, -1]

    # Compute Hamming distances
    n_correct = 0
    for d_i in range(len(subset)):
        idx, dist = net_sub.nearest_target(recon_sub[d_i])
        if idx == d_i:
            n_correct += 1

    acc = n_correct / len(subset) * 100
    axes[si].set_title(f'Digits {subset}\nAccuracy: {acc:.0f}%')

    n_show = min(len(subset), 5)
    for row, (data, lbl) in enumerate(zip(
        [targets_sub[:n_show], noisy_sub[:n_show], recon_sub[:n_show]],
        ['Original', 'Noisy', 'Reconstructed']
    )):
        for col in range(n_show):
            ax_img = axes[si].inset_axes(
                [col / n_show, 1 - (row + 1) / 3, 1 / n_show, 1 / 3],
            )
            ax_img.imshow(data[col].reshape(28, 28), cmap='gray', vmin=-1, vmax=1)
            ax_img.axis('off')
            if col == 0:
                ax_img.set_ylabel(lbl, fontsize=8)
    axes[si].axis('off')

plt.tight_layout()
plt.show()

### 3.6 Algorithm Comparison: Hebb vs LSSM on Digits

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ai, alg in enumerate(['Hebb', 'LSSM']):
    net_alg = HopfieldNetwork(all_targets_norm, alg=alg)
    noise_alg = 1.0
    noisy_alg = np.clip(
        all_targets_norm + rng.normal(scale=noise_alg, size=all_targets_norm.shape),
        -1, 1
    )
    states_alg, energies_alg = net_alg.simulate(noisy_alg, num_iter=30)
    final_alg = states_alg[:, :, -1]

    n_correct = sum(1 for d in range(10)
                    if net_alg.nearest_target(final_alg[d])[0] == d)

    axes[ai].set_title(f'{alg} Algorithm — {n_correct}/10 correct (σ={noise_alg})')
    for d in range(10):
        ax_img = axes[ai].inset_axes([d / 10, 0, 1 / 10, 1])
        ax_img.imshow(final_alg[d].reshape(28, 28), cmap='gray', vmin=-1, vmax=1)
        ax_img.axis('off')
        ax_img.set_title(f'{d}', fontsize=8)
    axes[ai].axis('off')

plt.suptitle('Reconstructed Digits: Hebb vs LSSM', fontsize=13)
plt.tight_layout()
plt.show()

### 3.7 Convergence Speed Analysis

In [ ]:
# How many iterations until convergence for each digit?
noise_conv = 1.0
noisy_conv = np.clip(
    all_targets_norm + rng.normal(scale=noise_conv, size=all_targets_norm.shape), -1, 1
)
states_conv, energies_conv = net_digits.simulate(noisy_conv, num_iter=50)

convergence_steps = []
for d in range(10):
    diffs = np.diff(energies_conv[d])
    # Find first step where energy change is negligible
    converged_at = 50
    for step in range(len(diffs)):
        if abs(diffs[step]) < 1e-6:
            converged_at = step + 1
            break
    convergence_steps.append(converged_at)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(range(10), convergence_steps, color='steelblue', edgecolor='black')
ax.set_xlabel('Digit')
ax.set_ylabel('Steps to Convergence')
ax.set_title(f'Convergence Speed per Digit (noise σ={noise_conv})')
ax.set_xticks(range(10))
ax.grid(True, alpha=0.3, axis='y')
for bar, steps in zip(bars, convergence_steps):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(steps), ha='center', fontsize=9)
plt.tight_layout()
plt.show()

---
## Exercise 4: Capacity and Spurious States

Systematically investigate the storage capacity of Hopfield networks and characterize what happens beyond capacity.

**The code below provides a comprehensive baseline.** You should analyze the results and discuss:
- Where does capacity break down empirically vs. theoretically?
- What types of spurious attractors appear?
- How does the LSSM algorithm compare to Hebbian learning?

### 4.1 Capacity Curve (Hebb Rule)

In [ ]:
dim = 100
max_patterns = 30
print(f"Running capacity test: D={dim}, max patterns={max_patterns}")
print(f"Theoretical Hebb capacity: ~{int(0.14 * dim)} patterns\n")

fig, ax = HopfieldNetwork.plot_capacity_analysis(
    dim=dim, max_patterns=max_patterns, trials=30, alg='Hebb'
)
plt.show()

### 4.2 Capacity Comparison: Hebb vs LSSM

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

HopfieldNetwork.plot_capacity_analysis(dim=50, max_patterns=20, trials=30, alg='Hebb', ax=ax1)
ax1.set_title('Capacity Analysis (Hebb, D=50)')

HopfieldNetwork.plot_capacity_analysis(dim=50, max_patterns=20, trials=30, alg='LSSM', ax=ax2)
ax2.set_title('Capacity Analysis (LSSM, D=50)')

plt.tight_layout()
plt.show()

### 4.3 Capacity Across Dimensions

How does the capacity threshold scale with network size? Test multiple dimensions and overlay the results.

In [ ]:
dims_to_test = [20, 50, 100]
fig, ax = plt.subplots(figsize=(10, 5))

for dim in dims_to_test:
    max_p = int(0.35 * dim)
    counts, rates = HopfieldNetwork.capacity_test(dim, max_p, trials=20, alg='Hebb')
    # Normalize x-axis by dimension to show the 0.14 threshold
    ax.plot(counts / dim, rates, 'o-', ms=3, lw=1.5, label=f'D={dim}')

ax.axvline(x=0.14, color='red', ls='--', lw=2, label='Theoretical: T/D = 0.14')
ax.set_xlabel('Load (T / D)')
ax.set_ylabel('Retrieval Success Rate')
ax.set_title('Capacity Scaling: Load vs Retrieval Rate')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 0.35)
plt.tight_layout()
plt.show()

### 4.4 Spurious Attractors Beyond Capacity

In [ ]:
# Store too many patterns in a small network and analyze spurious states
dim_sp = 20
n_patterns_sp = 8  # Well beyond capacity for D=20 (0.14*20 ≈ 3)

rng_sp = np.random.default_rng(42)
targets_sp = rng_sp.choice([-1.0, 1.0], size=(n_patterns_sp, dim_sp))
net_sp = HopfieldNetwork(targets_sp, alg='Hebb')
net_sp.summary()

# Check which stored patterns are still stable
print("\nStability of stored patterns:")
n_stable = 0
for i in range(n_patterns_sp):
    stable = net_sp.is_stable(targets_sp[i])
    if stable:
        n_stable += 1
    print(f"  Pattern {i}: stable = {stable}")
print(f"\n{n_stable}/{n_patterns_sp} patterns are stable fixed points")

# Find all fixed points
fps = net_sp.classify_fixed_points()
print(f"\nTotal fixed points found: {len(fps)}")

# Categorize them
n_stored = sum(1 for fp in fps
               if any(np.allclose(fp, t, atol=0.5) for t in targets_sp))
n_neg = sum(1 for fp in fps
            if any(np.allclose(fp, -t, atol=0.5) for t in targets_sp)
            and not any(np.allclose(fp, t, atol=0.5) for t in targets_sp))
n_spurious = len(fps) - n_stored - n_neg

print(f"  Stored patterns:     {n_stored}")
print(f"  Negation states:     {n_neg}")
print(f"  Spurious attractors: {n_spurious}")

In [ ]:
# Retrieval test at different overload levels
dim_ret = 30
overload_levels = [0.05, 0.10, 0.14, 0.20, 0.30, 0.50]
results = []

for load in overload_levels:
    n_pat = max(1, int(load * dim_ret))
    counts, rates = HopfieldNetwork.capacity_test(dim_ret, n_pat, trials=30, alg='Hebb')
    results.append((load, n_pat, rates[-1]))
    print(f"Load {load:.2f} ({n_pat} patterns): retrieval rate = {rates[-1]:.2%}")

print(f"\nThe network starts failing around load ≈ 0.14 as predicted by theory.")

---
## Exercise 5: Basins of Attraction Analysis

For the 3D Hopfield network from Exercise 2, exhaustively sample the state space and classify each initial state by which attractor it converges to.

**The code below provides the complete analysis pipeline.** Discuss:
- Which stored patterns have the largest basins?
- Do synchronous and asynchronous updates yield different basins?
- Are there spurious attractors, and what fraction of state space do they capture?

### 5.1 Programmatic Basin Mapping

In [ ]:
targets_3d = np.array([[1, 1, 1], [-1, -1, 1], [1, -1, -1]])
net_3d = HopfieldNetwork(targets_3d)

# Compute basins using the API (returns data, not just a plot)
init_pts, labels, attractors, fractions = net_3d.compute_basins(
    num_iter=30, sync=True
)

print("=== Basin Analysis (Synchronous) ===")
print(f"Sampled {len(init_pts)} initial states")
print(f"Found {len(attractors)} distinct attractors:\n")

for j, (att, frac) in enumerate(zip(attractors, fractions)):
    is_stored = any(np.allclose(att, t, atol=0.5) for t in targets_3d)
    is_neg = any(np.allclose(att, -t, atol=0.5) for t in targets_3d)
    label = "STORED" if is_stored else ("NEGATION" if is_neg else "SPURIOUS")
    stable = net_3d.is_stable(att)
    print(f"  Attractor {j}: {att}  [{label}]  basin={frac:.1%}  stable={stable}")

### 5.2 Sync vs Async Basin Comparison

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6),
                               subplot_kw={'projection': '3d'})

net_3d.plot_basins(resolution=15, num_iter=30, sync=True, ax=ax1)
ax1.set_title('Synchronous Update')

net_3d.plot_basins(resolution=15, num_iter=30, sync=False, ax=ax2)
ax2.set_title('Asynchronous Update')

fig.suptitle('Basins of Attraction: Sync vs Async', fontsize=14)
plt.tight_layout()
plt.show()

### 5.3 Multi-Trajectory Convergence

In [ ]:
starts_basin = [
    [0.5, 0.5, 0.5],
    [-0.5, -0.5, 0.5],
    [0.5, -0.5, -0.5],
    [0.0, 0.0, 0.0],
    [-1.0, 1.0, 1.0],
    [1.0, 1.0, -1.0],
    [-0.2, 0.8, -0.3],
    [0.9, -0.9, 0.9],
]

fig, axes = net_3d.plot_multi_trajectory(
    starts_basin, num_iter=20,
    labels=[f'({s[0]:+.1f},{s[1]:+.1f},{s[2]:+.1f})' for s in starts_basin]
)
plt.show()

### 5.4 Comprehensive Stability Report

In [ ]:
print("=" * 60)
print("  COMPREHENSIVE STABILITY REPORT")
print("=" * 60)

# All fixed points
fps = net_3d.classify_fixed_points()
spurious = net_3d.find_spurious_attractors()

print(f"\n  Network: D={net_3d.D}, T={net_3d.num_patterns}, alg={net_3d.alg}")
print(f"  Total fixed points: {len(fps)}")
print(f"  Spurious attractors: {len(spurious)}")
print()

for i, fp in enumerate(fps):
    is_stored = any(np.allclose(fp, t, atol=0.5) for t in targets_3d)
    is_neg = any(np.allclose(fp, -t, atol=0.5) for t in targets_3d)
    label = "STORED" if is_stored else ("NEGATION" if is_neg else "SPURIOUS")
    e = net_3d.energy(fp)
    idx, dist = net_3d.nearest_target(fp)
    print(f"  FP {i}: {fp}  [{label}]  E={e:.4f}  nearest_target=T{idx} (d={dist})")

# Basin sizes
print("\nBasin fractions:")
_, _, atts, fracs = net_3d.compute_basins(num_iter=30, sync=True)
for j, (a, f) in enumerate(zip(atts, fracs)):
    is_stored = any(np.allclose(a, t, atol=0.5) for t in targets_3d)
    label = "stored" if is_stored else "spurious"
    print(f"  Attractor {j}: {a}  ({label})  basin = {f:.1%}")

print("\n" + "=" * 60)

---

## Summary

| Exercise | Network | Key Insight |
|:---------|:--------|:------------|
| **1** | 2D, 3 targets | Complete toolkit tutorial — energy landscape, basins, fixed points, spurious attractors |
| **2** | 3D, 3 targets | Skeleton analysis — apply tools from Ex1 to investigate 3D state space |
| **3** | 784D, 10 digits | Comprehensive ablation — noise sensitivity, algorithm comparison, convergence analysis |
| **4** | Variable D | Capacity limits — experimental vs. theoretical, spurious states beyond capacity |
| **5** | 3D revisited | Basin mapping — sync vs async, attractor classification, stability report |

**Next steps:** Explore the interactive demos:
- `demos/hopfield_basin_explorer/app.py` — 3D energy landscape with MNIST digits
- `demos/modern_hopfield_attention/app.py` — Modern Hopfield Networks and Self-Attention